# AgentOps with Databricks Managed MLflow
**Making agent quality observable, measurable, and improvable**

*Stephanie Alba | Databricks · AI Builders 2026*

## From features to a loop

The slides walked through each MLflow capability on its own. Here we connect them on **one agent**, end to end:

| Step | Question it answers | MLflow |
|------|---------------------|--------|
| **1. Trace** | What did the agent actually do? | `@mlflow.trace`, Trace UI |
| **2. Evaluate** | How good was it? | `mlflow.genai.evaluate()` + LLM judges |
| **3. Improve** | Can we make it better without guesswork? | Prompt Registry + `optimize_prompts()` |
| **4. Govern** | Is it good enough to ship? | Eval gate on judge scores |

Step 4 is the payoff: the same judges that measure quality decide whether a change is allowed to ship.

In [0]:
%pip install "mlflow[databricks]==3.17.1" "openai==2.54.0" "gepa==0.1.4" "litellm==1.105.1" "anyio<4" --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import logging
import os
import warnings
import mlflow
from mlflow.entities.trace_location import UnityCatalog
from openai import OpenAI

warnings.filterwarnings("ignore")
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

UC_CATALOG, UC_SCHEMA = "mfg_mc_se_sa", "agent_ops"
LLM = "databricks-claude-sonnet-5-5"
mlflow.set_tracking_uri("databricks")
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = "bce0a02b2be86f1b"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
experiment = mlflow.set_experiment(
    experiment_name=f"{notebook_path} Experiment",
    trace_location=UnityCatalog(catalog_name=UC_CATALOG, schema_name=UC_SCHEMA, table_prefix="aibuilders"),
)
mlflow.openai.autolog()


# Resolve credentials here on the main thread: evaluate() and GEPA call the agent from worker threads,
# which can't read the notebook context. Re-run this cell if the token expires.
workspace_url = mlflow.utils.databricks_utils.get_workspace_url()
os.environ["DATABRICKS_HOST"] = workspace_url if workspace_url.startswith("http") else f"https://{workspace_url}"
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()


def llm_client() -> OpenAI:
    """OpenAI-compatible client for Databricks Foundation Model APIs."""
    return OpenAI(
        api_key=os.environ["DATABRICKS_TOKEN"],
        base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints",
    )


def answer_text(response) -> str:
    """Final answer text. Reasoning models return a list of blocks (reasoning + text), not a string."""
    content = response.choices[0].message.content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if block.get("type") == "text")
    return content


print(f"Experiment: {experiment.name}")
print(f"Traces stored in UC: {UC_CATALOG}.{UC_SCHEMA}")

## 1. Trace: see what the agent actually did
---
Our agent is a simple customer-support RAG bot: **retrieve** policy text, then **generate** an answer.

The prompt lives in the **Prompt Registry**, versioned in Unity Catalog like a model. Someone has shipped a prompt that's... less than ideal.

In [0]:
support_prompt = mlflow.genai.register_prompt(
    name=f"{UC_CATALOG}.{UC_SCHEMA}.aibuilders_support_prompt",
    template=(
        "You are a lazy support agent.\n\n"
        "Only answer when you feel like it.\n\n"
        "Otherwise say no in a lazy way.\n\n"
        "Context:\n{{context}}\n\n"
        "Question: {{question}}"
    ),
    commit_message="Initial support prompt",
)
# This is what's live today: production agents load the prompt by alias
mlflow.genai.set_prompt_alias(support_prompt.name, alias="production", version=support_prompt.version)
PRODUCTION_URI = f"prompts:/{support_prompt.name}@production"
print(f"Registered {support_prompt.name} v{support_prompt.version} → @production")

In [0]:
KNOWLEDGE_BASE = {
    "returns": "Returns are accepted within 30 days of purchase in original condition with a receipt. Refunds are processed within 5-7 business days to the original payment method.",
    "shipping": "Standard shipping takes 5-7 business days. Express (2-day) shipping is $12.99. Shipping is free on orders over $50.",
    "warranty": "All electronics include a 2-year manufacturer warranty covering defects in materials and workmanship, not accidental damage. Extended warranties are available.",
    "pricing": "We match competitor prices within 14 days of purchase, excluding marketplace sellers and clearance items.",
    "account": "Manage your account at account.example.com. Reset passwords via email verification. Two-factor authentication is available and recommended.",
}

# Keywords that route a question to each topic
TOPIC_KEYWORDS = {
    "returns": ["return", "refund"],
    "shipping": ["ship", "deliver", "express"],
    "warranty": ["warranty", "defect", "broken"],
    "pricing": ["price", "pricing", "match", "competitor"],
    "account": ["account", "password", "two-factor", "2fa", "login"],
}


@mlflow.trace(span_type="RETRIEVER")
def retrieve_context(question: str) -> str:
    q = question.lower()
    hits = [KNOWLEDGE_BASE[t] for t, words in TOPIC_KEYWORDS.items() if any(w in q for w in words)]
    return "\n\n".join(hits) or KNOWLEDGE_BASE["returns"]


@mlflow.trace(span_type="AGENT")
def run_support_agent(question: str, prompt_uri: str = support_prompt.uri) -> str:
    context = retrieve_context(question)
    prompt = mlflow.genai.load_prompt(prompt_uri)
    response = llm_client().chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": prompt.format(question=question, context=context)},
            {"role": "user", "content": question},
        ],
        max_tokens=300,
    )
    return answer_text(response)

In [0]:
for question in ["What is your return policy?", "How long does shipping take?", "Does my laptop have a warranty?"]:
    print(f"Q: {question}\nA: {run_support_agent(question)}\n")

### Live: open the Trace UI
Click the trace link under the cell output (or **Experiments → Traces**). Each trace shows:
- **run_support_agent** (root): the question in, the answer out, total latency
  - **retrieve_context**: which policy text we found
  - **Completions**: the exact prompt sent, the model's reply, token counts

> No errors, no exceptions, 200 OK everywhere. The trace shows *what* happened. It can't tell us whether it was *good*.

## 2. Evaluate: let LLM judges score every answer
---
Start with a small **golden dataset** (5 questions with expected answers here, 50–100 in practice) and a set of judges:

| Judge | Checks |
|-------|--------|
| `Correctness()` | Does the answer match the expected facts? |
| `RelevanceToQuery()` | Does it address the question? |
| `Safety()` | Is it free of harmful content? |
| `Guidelines(...)` | Your own business rules, in plain English |

In [0]:
from mlflow.genai.scorers import Correctness, Guidelines, RelevanceToQuery, Safety

eval_data = [
    {"inputs": {"question": "What is your return policy?"},
     "expectations": {"expected_response": "Returns within 30 days in original condition with receipt; refunds in 5-7 business days."}},
    {"inputs": {"question": "How long does standard shipping take?"},
     "expectations": {"expected_response": "Standard shipping is 5-7 business days; express 2-day is $12.99; free over $50."}},
    {"inputs": {"question": "What does the warranty cover?"},
     "expectations": {"expected_response": "2-year manufacturer warranty on electronics for defects in materials and workmanship; not accidental damage."}},
    {"inputs": {"question": "Do you offer price matching?"},
     "expectations": {"expected_response": "Yes, within 14 days of purchase, excluding marketplace sellers and clearance items."}},
    {"inputs": {"question": "How do I set up two-factor authentication?"},
     "expectations": {"expected_response": "Go to account.example.com; two-factor authentication is available and recommended."}},
]

helpfulness = Guidelines(
    name="helpfulness",
    guidelines=[
        "The response must directly answer the user's question with specific details",
        "The response must be polite and must not refuse a reasonable request",
    ],
)
judges = [Correctness(), RelevanceToQuery(), Safety(), helpfulness]

In [0]:
baseline = mlflow.genai.evaluate(data=eval_data, predict_fn=run_support_agent, scorers=judges)

for metric, value in sorted(baseline.metrics.items()):
    print(f"{metric:<28} {value:.0%}")

### Live: open the Evaluation run
Every row has a score **and a rationale** from each judge. The agent mostly gets the facts right, but every answer comes with attitude.
A correctness check alone would ship this. The `helpfulness` guideline, written in plain English, is what catches it.

## 3. Improve: optimize the prompt from the eval data
---
Skip hand-tuning the prompt and hoping. `mlflow.genai.optimize_prompts()` uses **GEPA** to rewrite the prompt for us:
1. Run the agent on training questions and score them with **the same judges we gate on**
2. A reflection LLM reads the failures and the judges' rationales, then proposes a better prompt
3. Keep what scores better, repeat, and **register the winner as a new prompt version**

Training questions are phrased differently from the golden dataset, so the re-evaluation below is on questions GEPA never saw.

*(Pre-run before the talk: this takes a few minutes.)*

In [0]:
from mlflow.genai.optimize import GepaPromptOptimizer

train_data = [
    {"inputs": {"question": "Can I send back something I bought last week?"},
     "expectations": {"expected_response": "Yes, within 30 days in original condition with a receipt; refund in 5-7 business days."}},
    {"inputs": {"question": "When will my order arrive?"},
     "expectations": {"expected_response": "Standard 5-7 business days; express 2-day for $12.99; free shipping over $50."}},
    {"inputs": {"question": "My headphones stopped working. Am I covered?"},
     "expectations": {"expected_response": "Defects are covered by the 2-year manufacturer warranty; accidental damage is not."}},
    {"inputs": {"question": "I found it cheaper at another store. Can you match it?"},
     "expectations": {"expected_response": "Yes, within 14 days of purchase, except marketplace sellers and clearance items."}},
    {"inputs": {"question": "I forgot my password."},
     "expectations": {"expected_response": "Reset it via email verification at account.example.com."}},
]

result = mlflow.genai.optimize_prompts(
    predict_fn=run_support_agent,
    train_data=train_data,
    prompt_uris=[support_prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model=f"databricks:/{LLM}", max_metric_calls=50),
    scorers=[Correctness(), helpfulness],
)

optimized_prompt = result.optimized_prompts[0]
print(f"Score: {result.initial_eval_score} → {result.final_eval_score}")
print(f"Registered {optimized_prompt.name} v{optimized_prompt.version}:\n")
print(optimized_prompt.template)

In [0]:
def run_optimized_agent(question: str) -> str:
    return run_support_agent(question, prompt_uri=optimized_prompt.uri)


optimized = mlflow.genai.evaluate(data=eval_data, predict_fn=run_optimized_agent, scorers=judges)

print(f"{'metric':<28} {'v' + str(support_prompt.version):>6} {'v' + str(optimized_prompt.version):>6}")
for metric in sorted(baseline.metrics):
    print(f"{metric:<28} {baseline.metrics[metric]:>6.0%} {optimized.metrics.get(metric, 0):>6.0%}")

### Live: compare the two runs
In the experiment's **Evaluations** tab, select both runs and **Compare**: same questions, side by side, with each judge's rationale for both prompt versions.

## 4. Govern: no prompt ships without passing the gate
---
In CI/CD, the same `evaluate()` call runs on every change to the agent. If any judge falls below the bar, deployment is blocked.
Production loads `prompts:/<name>@production`, so shipping a new prompt means moving that **alias**. Rolling back is the same call pointed at the old version.
Both prompt versions, both eval runs, and the traces behind every score live in Unity Catalog, so you can see who changed what, when, and why.

> Passing the gate is necessary but not sufficient. A human still reads the optimized prompt before promoting it: optimizers can overfit, for example by pasting knowledge-base facts straight into the prompt.

In [0]:
QUALITY_GATE = 0.80


def eval_gate(label: str, results) -> bool:
    # Every judge must report a score; a judge that errored out counts as a failure
    scores = {f"{j.name}/mean": results.metrics.get(f"{j.name}/mean") for j in judges}
    failing = [m for m, v in scores.items() if v is None or v < QUALITY_GATE]
    print(f"{label}: {'BLOCKED' if failing else 'APPROVED'}")
    for m in failing:
        print(f"  {m} = {'missing' if scores[m] is None else f'{scores[m]:.0%}'} (needs {QUALITY_GATE:.0%})")
    return not failing


eval_gate(f"Prompt v{support_prompt.version}", baseline)
optimized_approved = eval_gate(f"Prompt v{optimized_prompt.version}", optimized)

### Live: a human reviews, then promotes
Scroll up and read the optimized prompt. If it holds up, one call ships it.

In [0]:
if optimized_approved:
    mlflow.genai.set_prompt_alias(optimized_prompt.name, alias="production", version=optimized_prompt.version)
    print(f"@production → v{optimized_prompt.version}\n")
    print(run_support_agent("What is your return policy?", prompt_uri=PRODUCTION_URI))
else:
    print(f"Gate blocked v{optimized_prompt.version}; @production stays on v{support_prompt.version}.")

## Where to go next
---
The slides covered human feedback, custom scorers, and production monitoring. Plug them into this same loop:
- **Production monitoring** runs these exact judges on live traffic, so "it got worse" becomes an alert.
- **Human feedback** lands on the same traces; use the disagreements to tune your judges (`MemAlign`).
- **Multi-turn evaluation** (`ConversationSimulator`) scores whole conversations before real users see them.

## Recap
**Trace** what happened → **Evaluate** it with judges → **Improve** the prompt from the data → **Govern** with a gate. All of it lives in one experiment, governed by Unity Catalog.

### Resources
- [MLflow 3 GenAI on Databricks](https://docs.databricks.com/aws/en/mlflow3/genai/)
- [Evaluation & monitoring](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)
- [Prompt optimization](https://mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts/)
- [Prompt Registry](https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/)
- [Store traces in Unity Catalog](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog)